#Problem Statement: Build a Real-Time News Research Assistant Using LangChain Tools
Build a mini “News Research Assistant” using LangChain that can take a user’s query about current events (e.g. “Latest AI regulations in Europe”, “Current inflation news in India”, “Recent breakthroughs in cancer research”) and return a clear, structured summary using live web search.

Your assistant must use DuckDuckGoSearchResults as a tool and LangChain’s Runnable components.



 ### Install Requirements

In [23]:
!pip install langchain langchain-community langchain-google-genai duckduckgo-search -qU

In [24]:
!pip install -qU ddgs

###API Key Setup

In [3]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

###Imports

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough
import json
import re

###LLM & Search Tool Initialization

In [12]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

search_tool = DuckDuckGoSearchResults(output_format="list", num_results=8)

### Intent Extraction Prompt

In [13]:
intent_prompt = PromptTemplate.from_template("""
You are a news assistant. Given the user's query below, extract:
1. topic: A short phrase representing the main subject (e.g. "OpenAI", "India stock market")
2. type: One of — headlines, deep_dive, or mixed
   - headlines: user wants quick overview / bullet headlines
   - deep_dive: user wants detailed explanation or summary
   - mixed: user wants both headlines and a short summary

Respond ONLY in this exact JSON format:
{{"topic": "<short topic phrase>", "type": "<headlines|deep_dive|mixed>"}}

User query: {query}
""")

intent_chain = intent_prompt | llm | StrOutputParser()

def extract_intent(query: str) -> dict:
    raw = intent_chain.invoke({"query": query})

    raw = re.sub(r"```(?:json)?|```", "", raw).strip()
    return json.loads(raw)

print(extract_intent("What are today's top tech news headlines?"))

{'topic': 'tech news', 'type': 'headlines'}


###Fetch & Format Context

In [14]:
def fetch_and_format_context(topic: str, top_n: int = 5) -> str:
    results = search_tool.invoke(topic)
    top_results = results[:top_n]

    context_lines = []
    for i, item in enumerate(top_results, 1):
        title   = item.get("title", "No Title")
        snippet = item.get("snippet", item.get("body", "No description"))
        link    = item.get("link", item.get("url", "No link"))
        context_lines.append(
            f"Article {i}:\nHeadline: {title}\nDescription: {snippet}\nLink: {link}"
        )

    return "\n\n".join(context_lines)

ctx = fetch_and_format_context("Gemini news")
print(ctx[:500])

Article 1:
Headline: Official Gemini news and updates | Google Blog
Description: The latest news about Gemini . Chat to start writing, planning, learning and more with Google AI.
Link: https://blog.google/products-and-platforms/products/gemini/

Article 2:
Headline: Google News - Google's Gemini 3.1 Pro - Overview
Description: At Braze, Google Workspace with Gemini is powering faster insights, automated meeting notes, and smarter workflows—securely. See how they’re rethinking work with AI goo.gl


### Headlines, Summary & Analysis Prompts

In [15]:
headlines_prompt = PromptTemplate.from_template("""
You are a news assistant. Based on the articles below, generate 3–5 short bullet point headlines.

Rules:
- Each bullet is 1 sentence.
- Include a short label and brief explanation.
- Include the most relevant link in brackets at the end of each bullet.

Articles:
{context}

Topic: {topic}
""")

headlines_chain = headlines_prompt | llm | StrOutputParser()

deep_dive_prompt = PromptTemplate.from_template("""
You are a news assistant. Based on the articles below, write a short structured summary (2–4 paragraphs).

Cover:
1. What is happening
2. Why it matters
3. Key numbers, dates, or names

Style: beginner-friendly, clear, no fluff.

Articles:
{context}

Topic: {topic}
""")

deep_dive_chain = deep_dive_prompt | llm | StrOutputParser()

mixed_prompt = PromptTemplate.from_template("""
You are a news assistant. Based on the articles below:

1. First, give 3 short bullet-point headlines (1 sentence each, with link).
2. Then write 1–2 paragraphs summarizing the most important ideas.

Articles:
{context}

Topic: {topic}
""")

mixed_chain = mixed_prompt | llm | StrOutputParser()

###  Pipeline Function

In [17]:
def run_pipeline(query: str) -> str:

    intent = extract_intent(query)
    topic = intent["topic"]
    info_type = intent["type"]
    print(f"[Intent] topic='{topic}' | type='{info_type}'\n")

    context = fetch_and_format_context(topic)

    chain_input = {"topic": topic, "context": context}

    if info_type == "headlines":
        result = headlines_chain.invoke(chain_input)
    elif info_type == "deep_dive":
        result = deep_dive_chain.invoke(chain_input)
    else:
        result = mixed_chain.invoke(chain_input)

    return result

###  Testing Queries

In [18]:
query1 = "What are today's top tech news headlines?"
output1 = run_pipeline(query1)
print("=" * 60)
print(f"Query: {query1}")
print("=" * 60)
print(output1)

[Intent] topic='tech news' | type='headlines'

Query: What are today's top tech news headlines?
Here are 5 short bullet point headlines about tech news:

*   **Comprehensive Tech Updates:** Access the latest news, updates, and developments in technology, including breaking stories, innovations, and industry trends. [https://grokipedia.com/page/Latest_technology_news]
*   **Aggregated Tech News:** Browse full articles, watch videos, and explore thousands of titles on technology topics through a dedicated news platform. [https://news.google.com/topics/CAAqJggKIiBDQkFTRWdvSUwyMHZNRGRqTVhZU0FtVnVHZ0pWVXlnQVAB]
*   **Global Tech Coverage:** Find today's latest technology news from every corner of the globe, providing breaking international news coverage. [https://www.reuters.com/technology/]
*   **Startup & Innovation Focus:** Discover the newest startup and technology news, featuring insights from tech leaders and emerging companies. [https://techcrunch.com/]
*   **Future Tech & Culture:**

In [19]:
query2 = "What's happening with stock markets in India?"
output2 = run_pipeline(query2)
print("=" * 60)
print(f"Query: {query2}")
print("=" * 60)
print(output2)

[Intent] topic='India stock market' | type='mixed'

Query: What's happening with stock markets in India?
Here are 3 short bullet-point headlines:

*   NSE celebrates India's first listing on the Social Stock Exchange segment. [https://www.nseindia.com/?mos=0](https://www.nseindia.com/?mos=0)
*   Indian markets experience volatility, influenced by factors like F&O Expiry and Trump Tariff. [https://money.rediff.com/](https://money.rediff.com/)
*   Historical data reveals past stock market crashes in India characterized by rapid and substantial declines. [https://grokipedia.com/page/Stock_market_crashes_in_India](https://grokipedia.com/page/Stock_market_crashes_in_India)

The Indian stock market, primarily driven by the National Stock Exchange (NSE) and the Bombay Stock Exchange (BSE), is a dynamic environment offering live updates and various investment opportunities in stocks, ETFs, and IPOs. Recent market activity has shown volatility, with analysts attributing movements to factors suc

In [20]:
query3 = "Latest AI regulations in Europe"
output3 = run_pipeline(query3)
print("=" * 60)
print(f"Query: {query3}")
print("=" * 60)
print(output3)

[Intent] topic='AI regulations in Europe' | type='deep_dive'

Query: Latest AI regulations in Europe
The European Union (EU) has introduced a landmark set of rules known as the AI Act. This regulation is the world's first comprehensive legal framework specifically designed to govern artificial intelligence. Its main goal is to establish common standards for how AI systems are developed, used, and controlled across all EU member countries.

This new law is highly significant because it aims to make AI trustworthy and safe for everyone. By addressing the potential risks associated with AI, the EU seeks to ensure that these powerful technologies are used responsibly. This initiative also positions Europe as a global leader in setting ethical and practical standards for AI, hoping to influence how AI is managed not just within the EU but potentially worldwide, fostering trust in this rapidly evolving field.

The AI Act is a detailed 144-page regulation that was officially enacted by the EU